In [8]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from datetime import datetime

In [11]:
GROQ_API_KEY = "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"
PDF_PATH = "D:\\D\\work\\code\\2005.11401v4.pdf"


In [12]:
chunks = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=50).split_documents(
    PyPDFLoader(PDF_PATH).load()
)
vectorstore = FAISS.from_documents(
    chunks, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
print("✅ PDF Ready!\n")

C:\Users\Ali Mehdi\AppData\Local\Temp\ipykernel_7204\2676556210.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  chunks, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ PDF Ready!



In [ ]:

#_________Tools_________


def search_pdf(query: str) -> str:
    results = vectorstore.similarity_search(query, k=3)
    if not results:
        return "PDF mein kuch nahi mila."
    return "\n\n".join([d.page_content for d in results])

def calculator(expression: str) -> str:
    try:
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return "Sirf math expressions allowed hain."
        return f"Result: {eval(expression)}"
    except:
        return "Galat expression."

def get_time() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def detect_tool(user_input: str):
    text = user_input.lower()
    if any(op in text for op in ["+", "-", "*", "/"]):
        # expression extract karo
        import re
        expr = re.findall(r"[\d\+\-\*\/\(\)\. ]+", user_input)
        if expr:
            return "calculator", expr[0].strip()
    if any(w in text for w in ["time", "waqt", "date", "tarikh"]):
        return "time", None
    return "pdf", user_input

# === Agents ===
researcher = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)
writer     = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.4, api_key=GROQ_API_KEY)
checker    = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)

print("✅ Multi-Agent + Tools Ready!\n")

while True:
    user_input = input("You: ").strip()
    if not user_input: continue
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("👋 Goodbye!")
        break

    # === Tool Detection ===
    tool_name, tool_input = detect_tool(user_input)

    if tool_name == "calculator":
        print(f"🧮 Calculator: {calculator(tool_input)}\n")
        continue

    if tool_name == "time":
        print(f"🕐 Time: {get_time()}\n")
        continue

    # === PDF Query — 3 Agents ===
    print("🔍 Researcher kaam kar raha hai...")
    pdf_context = search_pdf(user_input)

    research = researcher.invoke([
        SystemMessage(content="Tum Researcher hu. PDF context se relevant info nikaal aur clearly explain kar."),
        HumanMessage(content=f"Sawal: {user_input}\n\nPDF:\n{pdf_context}")
    ])
    print("✅ Research Done!")

    print("✍️ Writer likh raha hai...")
    written = writer.invoke([
        SystemMessage(content="Tum Professional Writer hu. Research ko clean aur structured format mein likho."),
        HumanMessage(content=f"Sawal: {user_input}\n\nResearch:\n{research.content}")
    ])
    print("✅ Writing Done!")

    print("✔️ Checker check kar raha hai...")
    final = checker.invoke([
        SystemMessage(content="Tum Checker hu. Answer sahi hai toh APPROVED likho saath mein, galat hai toh fix karo."),
        HumanMessage(content=f"Sawal: {user_input}\n\nAnswer:\n{written.content}")
    ])

    print(f"\n✅ Final Answer:\n{final.content}\n")

✅ Multi-Agent + Tools Ready!

🕐 Time: 2026-05-19 17:07:51

🔍 Researcher kaam kar raha hai...
✅ Research Done!
✍️ Writer likh raha hai...
✅ Writing Done!
✔️ Checker check kar raha hai...

✅ Final Answer:
APPROVED

🔍 Researcher kaam kar raha hai...
✅ Research Done!
✍️ Writer likh raha hai...
✅ Writing Done!
✔️ Checker check kar raha hai...

✅ Final Answer:
APPROVED nahi, kyunki aapka pehla sawal nahi pucha gaya tha. Aapne kaha ki "maine kia pucha phle", lekin aapka koi pehla sawal nahi tha. Isliye, maine galat jawab diya hai.

Sahi jawab yeh hona chahiye:
"Aapka pehla sawal nahi pucha gaya hai. Kripya apna sawal puchhein, main aapko jawab dena chahta hoon."

🔍 Researcher kaam kar raha hai...
✅ Research Done!
✍️ Writer likh raha hai...
✅ Writing Done!
✔️ Checker check kar raha hai...

✅ Final Answer:
APPROVED 

Aapka jawab sahi hai aur aapne PDF mein diye gaye research paper ke mukhya binduon ko acchi tarah se samjhaya hai. Aapne RAG model ke components, uske training aur performance ke b